# Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

colab = True

import torch

cuda = torch.cuda.is_available()
cuda

# Install MATLAB in Colab

In [ ]:
wget https://www.mathworks.com/mpm/glnxa64/mpm && \
chmod +x mpm && \
./mpm install --release=R2025a --destination=/opt/matlab --products=MATLAB && \
ln -fs /opt/matlab/bin/matlab /usr/local/bin/matlab && \
MATLAB_DEPS_URL="https://raw.githubusercontent.com/mathworks-ref-arch/container-images/main/matlab-deps/r2025a/ubuntu22.04/base-dependencies.txt" && \
MATLAB_DEPENDENCIES=base-dependencies.txt && \
wget ${MATLAB_DEPS_URL} -O ${MATLAB_DEPENDENCIES} && \
xargs -a ${MATLAB_DEPENDENCIES} -r apt-get install --no-install-recommends -y && \
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:/opt/matlab/bin/glnxa64 && \
python3 -m pip install jupyter-matlab-proxy matlabengine==25.1.2

# Utils

In [3]:
import warnings
warnings.filterwarnings("ignore", module="tqdm")

import psutil
import subprocess
import os
from tqdm.auto import tqdm
import shutil
import gc
from os.path import join, exists
from PIL import Image
import matlab.engine
import numpy as np
from pathlib import Path

import zipfile
import tempfile
import time

import matplotlib.pyplot as plt
import time
from functools import wraps

In [4]:
from functools import wraps

def timer_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        execution_time = end_time - start_time
        print(f"Функция '{func.__name__}' выполнилась за {execution_time:0f} секунд")
        return result
    return wrapper



@timer_decorator
def cmd_zip(folder_path, zip_path, verbose=False):
    subprocess.run(
        f"zip -j -q {zip_path} {folder_path}/*",
        shell=True,
    )
    if verbose:
        print('zip done')

@timer_decorator
def cmd_unzip(folder_path, zip_path, verbose=False):
    subprocess.run(
        f"unzip -q -d {folder_path} -o {zip_path}",
        shell=True,
    )
    if verbose:
        print('unzip done')

@timer_decorator
def cmd_cp(source_zip_path, dest_zip_path, verbose=False):
    subprocess.run(
        f"cp {source_zip_path} {dest_zip_path}",
        shell=True
    )
    if verbose:
        print('cp done')

@timer_decorator
def cmd_rm_rf(folder_path, verbose=False):
    subprocess.run(
        f"rm -rf {folder_path}",
        shell=True
    )
    if verbose:
        print('rm done')

In [5]:
def plot_pgm_image(pillow_image, title='PGM Image'):
    plt.figure(figsize=(8, 6))
    plt.imshow(pillow_image, cmap='gray', vmin=0, vmax=255)
    plt.colorbar(label='Intensity')
    plt.title(title)

    plt.axis('off')  # Скрываем оси
    plt.show()

def plot_pgm_diff(pillow_stego_image, pillow_cover_image, title='embedding changes: +1 = white, -1 = black'):
    cover = np.array(pillow_cover_image.convert('L'))
    stego = np.array(pillow_stego_image.convert('L'))

    diff = np.abs(stego.astype(np.float64) - cover.astype(np.float64))

    plt.figure(figsize=(8, 6))
    plt.imshow(diff, cmap='hot', interpolation='nearest')
    plt.colorbar(label='Magnitude of change')
    plt.title(title)

    plt.axis('off')
    plt.show()

def title(text):
    print("\n" + "="*50)
    print(text)
    print("="*50)

In [6]:
def init_engine():
    eng = matlab.engine.start_matlab("-licmode onlinelicensing")

    root = '/content/drive/MyDrive/diplom/steganography'

    eng.addpath(join(root,'S-UNIWARD_matlab','matlab_MEX'), nargout=0)
    eng.addpath(join(root,'WOW_matlab','matlab_MEX'), nargout=0)
    eng.addpath(join(root,'HUGO_matlab','matlab','matlab_MEX'), nargout=0)
    eng.addpath(join(root,'MiPOD_matlab'), nargout=0)

    return eng

## ImageResizer

In [7]:
from PIL import Image
import random
from typing import Union, Tuple, Optional

class ImageResizer:
    """
    Модуль для изменения размера изображений с различными стратегиями.

    Поддерживаемые стратегии:
    - 'resize' - масштабирование изображения до целевого размера
    - 'center_crop' - вырезание центральной области целевого размера
    - 'random_crop' - вырезание случайной области целевого размера
    """

    def __init__(self, target_size: Union[int, Tuple[int, int]], strategy: str = 'resize'):
        """
        Args:
            target_size: Целевой размер. Может быть:
                        - int: (target_size, target_size) для квадрата
                        - tuple: (width, height)
            strategy: Стратегия изменения размера ('resize', 'center_crop', 'random_crop')
        """
        self.target_size = self._parse_size(target_size)
        self.strategy = strategy.lower()

        if self.strategy not in ['resize', 'center_crop', 'random_crop']:
            raise ValueError(f"Неизвестная стратегия: {strategy}. "
                           f"Доступны: 'resize', 'center_crop', 'random_crop'")

    def _parse_size(self, size: Union[int, Tuple[int, int]]) -> Tuple[int, int]:
        """Преобразует входной размер в кортеж (width, height)"""
        if isinstance(size, int):
            return (size, size)
        elif isinstance(size, (tuple, list)) and len(size) == 2:
            return (int(size[0]), int(size[1]))
        else:
            raise ValueError(f"Некорректный размер: {size}. Ожидается int или tuple (width, height)")

    def _get_crop_coords(self, image: Image.Image,
                         crop_width: int, crop_height: int,
                         random_crop: bool = False) -> Tuple[int, int, int, int]:
        """
        Вычисляет координаты для вырезания области

        Args:
            image: Исходное изображение
            crop_width: Ширина вырезаемой области
            crop_height: Высота вырезаемой области
            random_crop: Если True - случайное место, если False - центр

        Returns:
            Кортеж (left, top, right, bottom)
        """
        img_width, img_height = image.size

        # Проверяем, что изображение достаточно большое для вырезания
        if img_width < crop_width or img_height < crop_height:
            raise ValueError(f"Изображение ({img_width}x{img_height}) меньше целевого размера "
                           f"({crop_width}x{crop_height}) для обрезки")

        if random_crop:
            # Случайная позиция
            left = random.randint(0, img_width - crop_width)
            top = random.randint(0, img_height - crop_height)
        else:
            # Центральная позиция
            left = (img_width - crop_width) // 2
            top = (img_height - crop_height) // 2

        right = left + crop_width
        bottom = top + crop_height

        return (left, top, right, bottom)

    def resize(self, image: Image.Image) -> Image.Image:
        """
        Масштабирует изображение до целевого размера

        Args:
            image: PIL Image объект

        Returns:
            Измененное PIL Image
        """
        return image.resize(self.target_size, Image.Resampling.LANCZOS)

    def center_crop(self, image: Image.Image) -> Image.Image:
        """
        Вырезает центральную область целевого размера

        Args:
            image: PIL Image объект

        Returns:
            Обрезанное PIL Image
        """
        crop_coords = self._get_crop_coords(image, self.target_size[0], self.target_size[1],
                                           random_crop=False)
        return image.crop(crop_coords)

    def random_crop(self, image: Image.Image) -> Image.Image:
        """
        Вырезает случайную область целевого размера

        Args:
            image: PIL Image объект

        Returns:
            Обрезанное PIL Image
        """
        crop_coords = self._get_crop_coords(image, self.target_size[0], self.target_size[1],
                                           random_crop=True)
        return image.crop(crop_coords)

    def __call__(self, image: Image.Image) -> Image.Image:
        """
        Применяет выбранную стратегию к изображению

        Args:
            image: PIL Image объект

        Returns:
            Обработанное PIL Image
        """
        if self.strategy == 'resize':
            return self.resize(image)
        elif self.strategy == 'center_crop':
            return self.center_crop(image)
        elif self.strategy == 'random_crop':
            return self.random_crop(image)
        else:
            # Эта ошибка не должна возникнуть из-за проверки в __init__
            raise ValueError(f"Неизвестная стратегия: {self.strategy}")




# Test Matlab

In [ ]:
# Start engine
eng = init_engine()
eng.sqrt(4.0)

In [ ]:
filename = '1.pgm'
bpp = 0.2

dir = '/content/drive/MyDrive/diplom/images/BOSSbase/cover'
cover_image_path = join(dir, filename)
cover_image = Image.open(cover_image_path).convert('L')
cover_np = np.array(cover_image)

In [ ]:
# MATLAB API call
cover_matlab = matlab.uint8(cover_np.tolist())
payload =      matlab.single(bpp)

stego_matlab, distortion = eng.S_UNIWARD(cover_matlab, payload, nargout=2)

In [ ]:
stego_np = np.array(stego_matlab._data).reshape(stego_matlab.size[::1]).T
stego_image = Image.fromarray(stego_np.astype(np.uint8))

In [ ]:
plot_pgm_image(stego_image, title=f'Stego image ({bpp}bpp)')

plot_pgm_image(cover_image, title='Cover image')

plot_pgm_diff(stego_image, cover_image, title=f'Embedding difference ({bpp}bpp)')

# Use

In [11]:
# Start engine
eng = init_engine()
eng.sqrt(4.0)

In [8]:
########################################################
# Параметры генерации
########################################################
samples_num     = 10_000
bpp             = 0.4
target_size     = (512, 512)
dataset         = 'BOSSbase'
force_embed     = False
resize_strategy = 'resize'
algorithm_name  = 'hugo'
w_random_key = False
########################################################

h, w = target_size
param_folder_name = f'{h}_{w}_{resize_strategy}_{algorithm_name}_{bpp}bpp'

# Drive:
DRIVE_ROOT = '/content/drive/MyDrive/diplom/images'
DRIVE_DATASET = join(DRIVE_ROOT, dataset)
DRIVE_COVER   = join(DRIVE_DATASET, 'cover')
DRIVE_PARAM   = join(DRIVE_DATASET, param_folder_name)
DRIVE_PARAM_COVER = join(DRIVE_PARAM, 'cover')
DRIVE_PARAM_STEGO = join(DRIVE_PARAM, 'stego')

DRIVE_DATASET_cover_zip = join(DRIVE_DATASET, 'cover.zip')
DRIVE_PARAM_cover_zip = join(DRIVE_PARAM, 'cover.zip')
DRIVE_PARAM_stego_zip = join(DRIVE_PARAM, 'stego.zip')


os.makedirs(DRIVE_PARAM_COVER, exist_ok=True)
os.makedirs(DRIVE_PARAM_STEGO, exist_ok=True)

# Local:
LOCAL_ROOT = '/content/local_data'
LOCAL_DATASET     = join(LOCAL_ROOT, dataset)
LOCAL_COVER       = join(LOCAL_DATASET, 'cover')

LOCAL_PARAM       = join(LOCAL_DATASET, param_folder_name)
LOCAL_PARAM_COVER = join(LOCAL_PARAM, 'cover')
LOCAL_PARAM_STEGO = join(LOCAL_PARAM, 'stego')

LOCAL_DATASET_cover_zip = join(LOCAL_DATASET, 'cover.zip')
LOCAL_PARAM_cover_zip = join(LOCAL_PARAM, 'cover.zip')
LOCAL_PARAM_stego_zip = join(LOCAL_PARAM, 'stego.zip')

os.makedirs(LOCAL_COVER, exist_ok=True)
os.makedirs(LOCAL_PARAM_COVER, exist_ok=True)
os.makedirs(LOCAL_PARAM_STEGO, exist_ok=True)

In [9]:
# Cover: drive -> local
if not os.path.exists(DRIVE_DATASET_cover_zip):
    cmd_zip(DRIVE_COVER, DRIVE_DATASET_cover_zip)
if not os.path.exists(LOCAL_DATASET_cover_zip):
    cmd_cp(DRIVE_DATASET_cover_zip, LOCAL_DATASET_cover_zip)
if len(os.listdir(LOCAL_COVER)) == 0:
    cmd_unzip(LOCAL_COVER, LOCAL_DATASET_cover_zip)

In [ ]:
path_list = [join(LOCAL_COVER, f) for f in os.listdir(LOCAL_COVER)]

def matlab_api_call(algorithm_name, cover_matlab, payload, config=None):
    if config is not None:
        if algorithm_name == 's-uniward':
            stego_matlab, distortion = eng.S_UNIWARD(cover_matlab, payload, config, nargout=2)
        elif algorithm_name == 'wow':
            stego_matlab, distortion = eng.WOW(cover_matlab, payload, config, nargout=2)
        elif algorithm_name == 'hugo':
            stego_matlab, distortion = eng.HUGO_like(cover_matlab, payload, config, nargout=2)
        else:
            raise NotImplementedError(f'Для {algorithm_name} нет подходящего алгоритма встраивания!')
    else:
        if algorithm_name == 's-uniward':
            stego_matlab, distortion = eng.S_UNIWARD(cover_matlab, payload, nargout=2)
        elif algorithm_name == 'wow':
            stego_matlab, distortion = eng.WOW(cover_matlab, payload, nargout=2)
        elif algorithm_name == 'hugo':
            stego_matlab, distortion = eng.HUGO_like(cover_matlab, payload, nargout=2)
        else:
            raise NotImplementedError(f'Для {algorithm_name} нет подходящего алгоритма встраивания!')

    return stego_matlab, distortion

def process_image(path):
    filename = os.path.basename(path)
    cover_image_path = join(LOCAL_PARAM_COVER, filename)
    stego_image_path = join(LOCAL_PARAM_STEGO, filename)

    if (
        os.path.exists(stego_image_path)
        and os.path.exists(cover_image_path)
        and force_embed == False
    ):
        return

    cover_pil = Image.open(path).convert('L')
    if cover_pil.size != target_size:
        cover_pil_resized = ImageResizer(target_size=target_size, strategy=resize_strategy)(cover_pil)
    else:
        cover_pil_resized = cover_pil

    cover_np = np.array(cover_pil_resized)

    # MATLAB API call
    cover_matlab = matlab.uint8(cover_np.tolist())
    payload = matlab.single(bpp)

    if w_random_key:
        random_seed = random.randint(0, 2147483647)
        config = eng.struct()
        config['STC_h'] = matlab.uint32([10])
        config['seed'] = matlab.int32([random_seed])
        stego_matlab, distortion = matlab_api_call(algorithm_name, cover_matlab, payload, config)
    else:
        stego_matlab, distortion = matlab_api_call(algorithm_name, cover_matlab, payload)


    stego_np = np.array(stego_matlab._data).reshape(stego_matlab.size[::1]).T
    stego_image = Image.fromarray(stego_np.astype(np.uint8))

    cover_pil_resized.save(cover_image_path)
    stego_image.save(stego_image_path)

path_list = [join(LOCAL_COVER, f) for f in os.listdir(LOCAL_COVER)]

for i, path in enumerate(tqdm(path_list[:samples_num])):
    process_image(path)

    if i % 100 == 0:
        total_ram = psutil.virtual_memory().total / (1024**3)
        available_ram = psutil.virtual_memory().available / (1024**3)

        if available_ram < 20:
            eng = init_engine()



In [ ]:
# Cover prepared: local -> drive
cmd_zip(LOCAL_PARAM_COVER, LOCAL_PARAM_cover_zip)
cmd_cp(LOCAL_PARAM_cover_zip, DRIVE_PARAM_cover_zip)
cmd_rm_rf(DRIVE_PARAM_COVER)
cmd_unzip(DRIVE_PARAM_COVER, DRIVE_PARAM_cover_zip)
cmd_rm_rf(DRIVE_PARAM_cover_zip)

In [ ]:
# Stego prepared: local -> drive
cmd_zip(LOCAL_PARAM_STEGO, LOCAL_PARAM_stego_zip)
cmd_cp(LOCAL_PARAM_stego_zip, DRIVE_PARAM_stego_zip)
cmd_rm_rf(DRIVE_PARAM_STEGO)
cmd_unzip(DRIVE_PARAM_STEGO, DRIVE_PARAM_stego_zip)
cmd_rm_rf(DRIVE_PARAM_stego_zip)